# Article 1: selección, operador, expertise y soporte

Seleccionar `STAGE`: rq1, aggregation, expertise, support o controls. Las diferencias son emparejadas por seed. Las barras muestran desviación estándar, no equivalencia estadística. No usar el test para elegir retrospectivamente variantes o umbrales.


In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "article1/distillation.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from article1.analysis import load_results, comparisons, summarize, plot_effect, export
OUT = ROOT / "OUTPUTS/article1_v3"
STAGE = "rq1"
VERIFY_CACHES = False
context = load_results(OUT, stage=STAGE)
display(context["validation"])
if VERIFY_CACHES:
    from article1.audit import audit
    for filename, methods in context["input_blocks"]:
        report = audit(OUT / filename, source_root=OUT / "sources", methods=methods)
        if not report["ok"]:
            raise RuntimeError(report)
effects = comparisons(context)


In [ ]:
LABELS = {
    "selection_logit": "ORACLE-logit − FedDF-logit (pp)",
    "feddf_pooling": "FedDF-prob − FedDF-logit (pp)",
    "oracle_pooling": "ORACLE-prob − ORACLE-logit (pp)",
    "selection_prob": "ORACLE-prob − FedDF-prob (pp)",
    "expertise_gain": "EXPERT-prob − FedDF-prob (pp)",
    "oracle_expertise_gap": "ORACLE-prob − EXPERT-prob (pp)",
    "support": "EXPERT-prob-SR − EXPERT-prob (pp)",
    "consensus_logit": "Consensus-logit − FedDF-logit (pp)",
    "energy_logit": "Energy-logit − FedDF-logit (pp)",
}
tables, figures = {}, {}
for name, effect in effects.items():
    tables[name + "_paired"] = effect
    tables[name + "_summary"] = summarize(effect)
    display(LABELS[name], tables[name + "_summary"])
    figures[name] = plot_effect(effect, LABELS[name])
    display(figures[name])
    # Target changes are training diagnostics, not independent generalization.
    for metric in ("delta_target_nll", "delta_target_entropy"):
        tables[name + "_" + metric] = summarize(effect, column=metric)
        display(tables[name + "_" + metric])
export(OUT / "minimal_analysis" / STAGE, tables, figures)
plt.close("all")


## Interpretación

ORACLE selecciona aciertos por muestra, no expertise acreditada; EXPERT usa M y la etiqueta pública. ORACLE no es un upper bound garantizado del student. El contraste de operador mantiene el routing; cuando ORACLE no selecciona nadie, ambas variantes usan el mismo fallback FedDF-logit. No confundir ausencia de significación con equivalencia: fijar un margen práctico antes de interpretar resultados. Tres seeds ofrecen precisión limitada.

SR es una dimensión separada: probabilidades cero fuera del soporte y renormalización por teacher. No implica que la calidad del student mejore aunque aumente la probabilidad del target en la clase real. El diagnóstico focal EXPERT-logit y la temperatura se ejecutan por separado. Las fases supervisadas generan CSV; su análisis gráfico específico sigue pendiente.
